## Setup
This notebook uses the local `.venv` virtual environment.  
**Select kernel:** `Python (Spradley)` in the top-right kernel picker before running.

In [ ]:
# ── C0: Config ───────────────────────────────────────────────────────────────
# Edit the values below before running. These override pipeline_core.py defaults.
# If you skip this cell, pipeline_core.py defaults apply automatically.

import pipeline_core

# ── Input data ───────────────────────────────────────────────────────────────
# INPUT_FILE: filename inside interview_input/
# INPUT_FORMAT: "spradley_v2" for Spradley platform exports
#               "legacy"      for the original pilot_transcripts.csv format
pipeline_core.CONFIG["INPUT_FILE"]   = "the-office-2.csv"
pipeline_core.CONFIG["INPUT_FORMAT"] = "spradley_v2"

# ── Tunable per run — edit here ──────────────────────────────────────────────
pipeline_core.CONFIG["LLM_MODEL"]       = "claude-haiku-4-5-20251001"
pipeline_core.CONFIG["LLM_TEMPERATURE"] = 0.2
pipeline_core.CONFIG["L2_CODES_RANGE"]  = (10, 20)  # codes per interview
pipeline_core.CONFIG["L3_CODES_RANGE"]  = (30, 60)  # consolidated codes across all employees
pipeline_core.CONFIG["CLUSTERS_RANGE"]  = (4, 6)    # thematic clusters

print("Config active:")
print(f"  Input:       {pipeline_core.CONFIG['INPUT_FILE']}  (format={pipeline_core.CONFIG['INPUT_FORMAT']})")
print(f"  Model:       {pipeline_core.CONFIG['LLM_MODEL']}  (temp={pipeline_core.CONFIG['LLM_TEMPERATURE']})")
print(f"  L2 range:    {pipeline_core.CONFIG['L2_CODES_RANGE']}")
print(f"  L3 range:    {pipeline_core.CONFIG['L3_CODES_RANGE']}")
print(f"  Clusters:    {pipeline_core.CONFIG['CLUSTERS_RANGE']}")

In [27]:
# ── C1: Imports & env ────────────────────────────────────────────────────────
import os
import json
from collections import defaultdict
from dotenv import load_dotenv
import pipeline_core

load_dotenv(pipeline_core.KEYS_ENV)

loaded_keys = [k for k in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY"] if os.getenv(k)]
print(f"Env loaded from '{pipeline_core.KEYS_ENV}'")
print(f"Keys found: {loaded_keys if loaded_keys else 'none — check keys.env'}")

Env loaded from 'keys.env'
Keys found: ['ANTHROPIC_API_KEY']


In [28]:
# ── C2: LLM client ───────────────────────────────────────────────────────────
# Defined in pipeline_core.py (search "C2" there to find/edit the implementation).
# To swap providers: edit pipeline_core.CONFIG["LLM_PROVIDER"] in C0 above.
# verify=False   — bypasses corporate TLS inspection (API key still authenticates).
# trust_env=False — ignores any HTTPS_PROXY / HTTP_PROXY env vars; connect directly.

from pipeline_core import call_llm

cfg = pipeline_core.CONFIG
print(f"LLM client ready: {cfg['LLM_PROVIDER']} / {cfg['LLM_MODEL']}  (temperature={cfg['LLM_TEMPERATURE']})")

LLM client ready: anthropic / claude-haiku-4-5-20251001  (temperature=0.2)


In [ ]:
# ── C3: Data ingestion ───────────────────────────────────────────────────────
# Defined in pipeline_core.py (search "C3"). Edit there to adapt to future input formats.

from pipeline_core import load_interviews

interviews = load_interviews()

print(f"Loaded {len(interviews)} interviews\n")
for iv in interviews:
    print(f"  {iv['interview_id'][:8]}...  {len(iv['qa_pairs'])} Q&A pairs")

print("\nSample (first interview, first 2 pairs):")
for qa in interviews[0]["qa_pairs"][:2]:
    print(f"  [t{qa['turn_number']}] Q: {qa['question'][:65]}")
    print(f"         A: {qa['answer'][:65]}")
    print()

In [30]:
# ── C4: Anonymizer ───────────────────────────────────────────────────────────
# Defined in pipeline_core.py (search "C4"). PII patterns are editable there.

from pipeline_core import anonymize

sample_raw  = interviews[0]["qa_pairs"][0]["answer"]
sample_anon = anonymize(sample_raw)
print("Anonymizer ready.")
print(f"  Before: {sample_raw[:80]}")
print(f"  After:  {sample_anon[:80]}")

Anonymizer ready.
  Before: Great team and exciting new things coming up!
  After:  Great team and exciting new things coming up!


In [31]:
# ── C5: DB init (L1) ─────────────────────────────────────────────────────────
# Builds the primary Q&A fact table. PK = interview_question_id (e.g. "6abc1e3b_t1").
# Shape is fixed after this cell; downstream cells read but do not modify db.

db = {}
for iv in interviews:
    prefix = iv["interview_id"][:8]
    for qa in iv["qa_pairs"]:
        iq_id = f"{prefix}_t{qa['turn_number']}"
        db[iq_id] = {
            "interview_id":      iv["interview_id"],
            "turn_number":       qa["turn_number"],
            "question":          qa["question"],
            "anonymised_answer": anonymize(qa["answer"]),
        }

print(f"DB initialised: {len(db)} entries")
sample_key = next(iter(db))
print(f"\nSample entry  ({sample_key}):")
for k, v in db[sample_key].items():
    print(f"  {k}: {str(v)[:80]!r}")

DB initialised: 49 entries

Sample entry  (6abc1e3b_t1):
  interview_id: '6abc1e3b-0f8a-423d-aed6-6b84640dcda1'
  turn_number: '1'
  question: '[initial mood opener]'
  anonymised_answer: 'Great team and exciting new things coming up!'


In [ ]:
# ── C6: L2 per-interview coding ───────────────────────────────────────────────
# One LLM call per interview. The model sees all Q&A turns in sequence and returns
# 20-30 thematic codes with the turn IDs that support each code.
# Prompt defined in pipeline_core.py (search "C6").

from pipeline_core import code_one_interview

interview_store = {}

by_interview = defaultdict(list)
for iq_id, entry in db.items():
    by_interview[entry["interview_id"]].append((iq_id, entry))

for interview_id, id_entry_pairs in by_interview.items():
    prefix = interview_id[:8]
    qa_entries = [
        (iq_id, e["question"], e["anonymised_answer"])
        for iq_id, e in sorted(id_entry_pairs, key=lambda x: int(x[1]["turn_number"]))
    ]
    l2_codes = code_one_interview(prefix, qa_entries)
    interview_store[interview_id] = {"l2_codes": l2_codes}
    print(f"  {prefix}:  {len(l2_codes)} L2 codes")

print(f"\nL2 coding done: {len(interview_store)} interviews")
sample_iv = next(iter(interview_store))
print(f"\nSample L2 codes ({sample_iv[:8]}...):")
for item in interview_store[sample_iv]["l2_codes"][:3]:
    print(f"  {item['code']!r:42}  <- {item['source_qa_ids']}")

In [ ]:
# ── C8: Global Consolidator (L3) ─────────────────────────────────────────────
# All L2 codes → L3_CODES_RANGE consolidated codes with merge lineage.
# Prompt defined in pipeline_core.py (search "C8").

from pipeline_core import consolidate_l3

n_interviews = len(interview_store)
all_l2_codes = [item["code"] for store in interview_store.values() for item in store["l2_codes"]]

global_store = {"l3_codes": consolidate_l3(all_l2_codes, n_interviews)}

n_l3 = len(global_store["l3_codes"])
l3_range = pipeline_core.CONFIG["L3_CODES_RANGE"]
print(f"L3 consolidation done: {n_l3} codes  (range: {l3_range[0]}-{l3_range[1]})")
print("\nFirst 10 L3 codes:")
for item in global_store["l3_codes"][:10]:
    print(f"  {item['code']!r:42}  <- {item['merged_from_l2']}")

In [ ]:
# ── C8b: Theme Clustering + Lineage ──────────────────────────────────────────
# L3 codes → named clusters. Prompt defined in pipeline_core.py (search "C8").
# Lineage (PK = cluster name) is built in this cell — it is orchestration, not a function.

from pipeline_core import cluster_l3_codes

l3_list     = [item["code"] for item in global_store["l3_codes"]]
cluster_map = cluster_l3_codes(l3_list)

# ── Build lookup tables ───────────────────────────────────────────────────────
l3_to_l2 = {item["code"]: item["merged_from_l2"] for item in global_store["l3_codes"]}

l2_to_interview = {}
l2_to_qa_ids    = {}
for interview_id, store in interview_store.items():
    for item in store["l2_codes"]:
        l2_to_interview[item["code"]] = interview_id
        l2_to_qa_ids[item["code"]]    = item["source_qa_ids"]

# ── Build lineage ─────────────────────────────────────────────────────────────
lineage  = {}
clusters = {}

for name, l3_codes_in_cluster in cluster_map.items():
    l2_entries, qa_ids = [], []
    for l3_code in l3_codes_in_cluster:
        for l2_code in l3_to_l2.get(l3_code, []):
            interview_id  = l2_to_interview.get(l2_code, "unknown")
            source_qa_ids = l2_to_qa_ids.get(l2_code, [])
            l2_entries.append({"code": l2_code, "interview_id": interview_id,
                               "source_qa_ids": source_qa_ids})
            qa_ids.extend(source_qa_ids)

    lineage[name] = {
        "l3_codes":  l3_codes_in_cluster,
        "l2_codes":  l2_entries,
        "l1_qa_ids": list(dict.fromkeys(qa_ids)),
    }
    clusters[name] = {"l3_codes": l3_codes_in_cluster, "story": ""}

cl_range = pipeline_core.CONFIG["CLUSTERS_RANGE"]
print(f"Clustering done: {len(clusters)} clusters (range: {cl_range[0]}-{cl_range[1]})\n")
for name, lin in lineage.items():
    print(f"  {name}")
    print(f"    Sources: {lin['l1_qa_ids'][:5]}{'...' if len(lin['l1_qa_ids']) > 5 else ''}")

In [ ]:
# ── C9: LLM Explainer ────────────────────────────────────────────────────────
# Per cluster: structured finding (category, tagline, summary, quotes, tag).
# Prompts defined in pipeline_core.py (search "C9").

from pipeline_core import explain_cluster, propose_experiments

for name, lin in lineage.items():
    iq_ids = lin["l1_qa_ids"]

    qa_lines = []
    for iq_id in iq_ids:
        if iq_id in db:
            e = db[iq_id]
            qa_lines.append(f"Q: {e['question']}\nA: {e['anonymised_answer']}")

    qa_pairs_text = "\n\n".join(qa_lines) if qa_lines else "(no source answers found)"
    result        = explain_cluster(name, clusters[name]["l3_codes"], qa_pairs_text)
    voice_count   = len({e["interview_id"] for e in lin["l2_codes"]})

    clusters[name].update({
        "category":    result.get("category", "mixed"),
        "tagline":     result.get("tagline", ""),
        "summary":     result.get("summary", ""),
        "quotes":      result.get("quotes", []),
        "tag":         result.get("tag", ""),
        "story":       result.get("summary", ""),
        "voice_count": voice_count,
    })
    print(f"[{result.get('category', '?')}]  {name}  ·  {result.get('tag', '')}  ·  {voice_count} voices")

# ── Experiment proposals ──────────────────────────────────────────────────────
needs_attention = [
    (name, data) for name, data in clusters.items()
    if data.get("category") in ("needs_work", "mixed")
]
experiments = propose_experiments(needs_attention)

print(f"\nExperiment proposals: {len(experiments)}")
for exp in experiments:
    print(f"  · {exp['title']}  ({exp.get('tag', '')})")

In [ ]:
# ── C9b: Headline narrative ───────────────────────────────────────────────────
# Generates the report opening and closing from all cluster findings.
# Run after C9 (needs tagline + summary on every cluster).

from pipeline_core import generate_headline

print("Generating headline narrative and closing note...")
headline_data = generate_headline(
    clusters,
    n_interviews=len(interviews),
    model=pipeline_core.CONFIG["LLM_MODEL"],
)
global_store["headline"]        = headline_data.get("headline", "")
global_store["note_to_protect"] = headline_data.get("note_to_protect", "")

print("\n--- The headline ---")
print(global_store["headline"])
print("\n--- A note on what to protect ---")
print(global_store["note_to_protect"])

In [37]:
# ── C11: Persist ─────────────────────────────────────────────────────────────
# Write all data structures to OUTPUT_DIR as pretty-printed JSON.
# Re-run after any cell to checkpoint progress.

import datetime

os.makedirs(pipeline_core.OUTPUT_DIR, exist_ok=True)

to_save = {
    "db":              db,
    "interview_store": interview_store,
    "global_store":    global_store,
    "lineage":         lineage,
    "clusters":        clusters,
    "experiments":     globals().get("experiments", []),
}

for name, data in to_save.items():
    path = os.path.join(pipeline_core.OUTPUT_DIR, f"{name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    size_kb = os.path.getsize(path) / 1024
    print(f"  Saved: {path}  ({size_kb:.1f} KB)")

# ── run_meta.json — attributes results to this model run ──────────────────────
run_meta = {
    "model":        pipeline_core.CONFIG["LLM_MODEL"],
    "temperature":  pipeline_core.CONFIG["LLM_TEMPERATURE"],
    "timestamp":    datetime.datetime.utcnow().isoformat() + "Z",
    "n_interviews": len(interviews),
}
meta_path = os.path.join(pipeline_core.OUTPUT_DIR, "run_meta.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(run_meta, f, indent=2)
print(f"  Saved: {meta_path}")

print(f"\nAll files written to '{pipeline_core.OUTPUT_DIR}'")

  Saved: pipeline_output\db.json  (16.1 KB)
  Saved: pipeline_output\interview_store.json  (11.8 KB)
  Saved: pipeline_output\global_store.json  (10.7 KB)
  Saved: pipeline_output\lineage.json  (21.2 KB)
  Saved: pipeline_output\clusters.json  (28.9 KB)
  Saved: pipeline_output\experiments.json  (4.1 KB)
  Saved: pipeline_output\run_meta.json

All files written to 'pipeline_output'


C:\Users\t02mham\AppData\Local\Temp\ipykernel_6724\3736731106.py:29: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":    datetime.datetime.utcnow().isoformat() + "Z",


In [ ]:
# ── C12: Report renderer ─────────────────────────────────────────────────────
# Renders the full HR report as styled HTML directly in the notebook output.
# Implementation lives in pipeline_core.py (search "C12 / C13").

from IPython.display import display, HTML
from pipeline_core import build_inline_report_html

display(HTML(build_inline_report_html(clusters, interviews, experiments)))

In [ ]:
# ── C13: Web application ─────────────────────────────────────────────────────
# Writes pipeline_output/report.html and opens it in the default browser.
# Tab 1 — Report: findings grouped by category, expandable summaries + quotes, experiments.
# Tab 2 — Lineage: fully collapsible tree  cluster -> L3 -> L2 -> L1 -> source Q&A.
# Implementation lives in pipeline_core.py (search "C12 / C13").

import webbrowser
from pipeline_core import build_report_html

_out_path = os.path.join(pipeline_core.OUTPUT_DIR, "report.html")
with open(_out_path, "w", encoding="utf-8") as _f:
    _f.write(build_report_html(clusters, interviews, experiments,
                               global_store, interview_store, lineage, db))

_abs = os.path.abspath(_out_path)
print(f"Written: {_abs}")
webbrowser.open("file:///" + _abs.replace(os.sep, "/"))
print("Opened in browser.")